# Laboratorio 20. Redes para datos tabulares con embeddings: fuga de clientes

**Pregunta guía:** ¿Qué aporta una red con *embeddings* de variables categóricas frente a una regresión logística con *one-hot* para priorizar clientes en riesgo de fuga?

## Objetivos
Al finalizar podrás codificar variables categóricas como índices con una categoría reservada para valores desconocidos; entrenar una regresión logística y una red con embeddings en PyTorch; usar *early stopping* con validación; evaluar con AUC, *log loss* y precisión en el 10 % superior; e interpretar los embeddings aprendidos.

## Prerrequisitos y conexión
Usa el ciclo de entrenamiento (laboratorio 08), la regularización (laboratorio 11) y la evaluación con train, validación y test del proyecto integrador (laboratorio 19). Cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
Una capa `nn.Embedding` es una tabla de parámetros: la categoría $c$ se representa con el vector aprendible $e_c \in \mathbb{R}^d$. Es equivalente a multiplicar un vector *one-hot* por una matriz, pero sin construir el vector disperso. Con $d$ pequeño, categorías de comportamiento parecido terminan con vectores cercanos, lo que permite compartir información entre categorías con pocos datos. La red concatena los embeddings con las variables numéricas estandarizadas y estima $P(\text{fuga}=1)=\sigma(f_\theta(x))$ minimizando entropía cruzada binaria.

## Problema y datos
Una empresa de servicios ficticia quiere priorizar una campaña de retención con capacidad para contactar al **10 %** de sus clientes. Generamos 20 000 clientes con plan, región, canal de adquisición y una de 40 sucursales, más antigüedad, gasto, reclamos y uso de la app. Cada sucursal tiene un riesgo latente no observado, los reclamos pesan más en planes altos (una interacción), el riesgo sube bruscamente al acercarse la renovación anual del contrato y cuando el cliente casi no usa la app (efectos no lineales). Los meses hasta la renovación son una variable construida con conocimiento del negocio. Todos los datos se generan en el notebook.

## Perspectiva de negocio y datos
Para la oficina de datos, el modelo es un activo que debe documentar qué variables usa, cómo trata categorías nuevas (una sucursal que abre mañana) y con qué métrica se decide. La métrica relevante no es la exactitud: es cuántos clientes que efectivamente se van quedan dentro de la capacidad de contacto.

## Experimento conceptual
¿Mejora la red con embeddings la priorización respecto de la regresión logística con la misma información? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional. Usamos `rng` (generador de NumPy) para los datos y `torch.manual_seed` antes de cada entrenamiento para que las comparaciones entre modelos sean justas.

In [ ]:
import copy
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 2026
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)

**Resultado e interpretación.** La celda confirma el entorno utilizado. El laboratorio está diseñado para CPU y se ejecuta en pocos minutos; una GPU disponible se utiliza sin ser requisito.

### Generar clientes y particiones
Simulamos el proceso generador con efectos conocidos. Separamos train, validación y test **antes** de cualquier transformación.

In [ ]:
n_customers = 20000
plan_names = ['basico', 'estandar', 'premium', 'empresa']
region_names = ['Norte', 'Centro', 'Metropolitana', 'Valparaiso', 'Sur', 'Austral']
channel_names = ['tienda', 'web', 'app', 'telefono', 'partner']
n_branches = 40
branch_risk = rng.normal(0.0, 0.8, n_branches)  # efecto latente por sucursal: el modelo no lo observa

plan = rng.choice(4, n_customers, p=[0.40, 0.30, 0.20, 0.10])
region = rng.choice(6, n_customers, p=[0.12, 0.15, 0.38, 0.13, 0.15, 0.07])
channel = rng.choice(5, n_customers, p=[0.35, 0.25, 0.20, 0.12, 0.08])
branch = rng.integers(0, n_branches, n_customers)
tenure_months = rng.gamma(2.0, 12.0, n_customers).clip(1, 120)
monthly_spend = (15000 + 12000 * plan + rng.normal(0, 4000, n_customers)).clip(5000)
complaints_90d = rng.poisson(0.5 + 0.4 * (plan == 0), n_customers)
app_days_30d = rng.binomial(30, np.where(channel == 2, 0.6, 0.3))

months_to_renewal = 12 - tenure_months % 12  # contratos anuales: variable aportada por el área comercial
logit = (-2.0 - 0.02 * tenure_months + 3.0 * np.exp(-months_to_renewal / 1.5)
         + 0.2 * complaints_90d + 1.5 * complaints_90d * (plan >= 2)
         + 2.0 * np.exp(-app_days_30d / 3) + branch_risk[branch]
         + np.array([0.4, 0.0, -0.3, -0.8])[plan]
         + np.array([0.1, 0.0, 0.2, 0.0, -0.1, 0.3])[region]
         + np.array([0.0, 0.2, -0.2, 0.3, 0.5])[channel])
churn = (rng.random(n_customers) < 1 / (1 + np.exp(-logit))).astype(np.float32)

order = rng.permutation(n_customers)
train_idx, val_idx, test_idx = order[:12000], order[12000:16000], order[16000:]
print('tasa de fuga global:', round(churn.mean(), 3))
print('train/val/test:', len(train_idx), len(val_idx), len(test_idx),
      '| tasa por split:', [round(float(churn[idx].mean()), 3) for idx in (train_idx, val_idx, test_idx)])

**Resultado e interpretación.** La tasa de fuga ronda un quinto de los clientes y es parecida en los tres splits. Con clases desbalanceadas, un modelo que predice "no se va" para todos tendría alta exactitud y ningún valor para la campaña: por eso usaremos métricas de ordenamiento.

### Codificar categorías y estandarizar numéricas
Cada categoría se convierte en un índice entero. El índice **0 queda reservado para "desconocido"**, de modo que una sucursal nueva en producción tenga una representación definida. Media y desviación se calculan solo con train.

In [ ]:
categorical = {'plan': plan, 'region': region, 'channel': channel, 'branch': branch}
cardinalities = [len(plan_names) + 1, len(region_names) + 1, len(channel_names) + 1, n_branches + 1]
X_cat = np.stack([values + 1 for values in categorical.values()], axis=1)  # 0 = desconocido

numeric = np.stack([tenure_months, months_to_renewal, monthly_spend, complaints_90d, app_days_30d],
                   axis=1).astype(np.float32)
numeric_mean, numeric_std = numeric[train_idx].mean(axis=0), numeric[train_idx].std(axis=0)
X_num = (numeric - numeric_mean) / numeric_std

def to_tensors(idx):
    return (torch.tensor(X_cat[idx], dtype=torch.long, device=device),
            torch.tensor(X_num[idx], dtype=torch.float32, device=device),
            torch.tensor(churn[idx], dtype=torch.float32, device=device))

splits = {'train': to_tensors(train_idx), 'val': to_tensors(val_idx), 'test': to_tensors(test_idx)}
print('cardinalidades (incluye desconocido):', dict(zip(categorical, cardinalities)))
print('X_cat:', X_cat.shape, '| X_num:', X_num.shape)
print('medias numéricas en train tras estandarizar:', X_num[train_idx].mean(axis=0).round(3))

**Resultado e interpretación.** Las numéricas quedan centradas en train (media cercana a cero); validación y test usan los mismos parámetros, por lo que no filtran información. Con *one-hot*, la sucursal ocuparía 41 columnas; con un embedding de dimensión 4 ocupa 4 parámetros por categoría.

### Métricas de priorización y baseline
Implementamos AUC (probabilidad de que un cliente que se fue reciba mayor puntaje que uno que se quedó), *log loss* y precisión en el 10 % de mayor riesgo. El baseline asigna a todos la tasa de fuga de train.

In [ ]:
def roc_auc(y_true, scores):
    ranks = np.empty(len(scores))
    ranks[np.argsort(scores)] = np.arange(1, len(scores) + 1)
    n_pos = y_true.sum()
    n_neg = len(y_true) - n_pos
    return (ranks[y_true == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def log_loss(y_true, probabilities):
    p = np.clip(probabilities, 1e-7, 1 - 1e-7)
    return float(-np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)))


def precision_at_top(y_true, scores, fraction=0.10):
    k = int(len(scores) * fraction)
    return float(y_true[np.argsort(-scores)[:k]].mean())


def report(y_true, probabilities):
    return {'log_loss': round(log_loss(y_true, probabilities), 4),
            'auc': round(float(roc_auc(y_true, probabilities)), 4),
            'precision_top10': round(precision_at_top(y_true, probabilities), 4)}


baseline_probability = np.full(len(val_idx), churn[train_idx].mean())
print('baseline en validación:', {'log_loss': round(log_loss(churn[val_idx], baseline_probability), 4),
                                   'auc': 0.5, 'precision_top10': round(float(churn[val_idx].mean()), 4)})

**Resultado e interpretación.** El baseline no ordena clientes: su AUC es 0,5 y su precisión en el 10 % superior es la tasa base. Cualquier modelo útil debe superar esos valores; la razón entre la precisión del modelo y la tasa base es el *lift* de la campaña.

### Definir los modelos
La regresión logística recibe *one-hot* más numéricas: un efecto aditivo por categoría y sin interacciones. La red recibe embeddings más numéricas y dos capas ocultas, lo que le permite combinar variables (por ejemplo, reclamos y plan).

In [ ]:
class OneHotLogistic(nn.Module):
    def __init__(self, cardinalities, n_numeric):
        super().__init__()
        self.cardinalities = cardinalities
        self.linear = nn.Linear(sum(cardinalities) + n_numeric, 1)

    def forward(self, x_cat, x_num):
        one_hot = [F.one_hot(x_cat[:, i], c).float() for i, c in enumerate(self.cardinalities)]
        return self.linear(torch.cat(one_hot + [x_num], dim=1)).squeeze(1)


class EmbeddingMLP(nn.Module):
    def __init__(self, cardinalities, n_numeric, embedding_dims, hidden=64, dropout=0.2):
        super().__init__()
        self.embeddings = nn.ModuleList(nn.Embedding(c, d) for c, d in zip(cardinalities, embedding_dims))
        self.mlp = nn.Sequential(
            nn.Linear(sum(embedding_dims) + n_numeric, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))

    def forward(self, x_cat, x_num):
        embedded = [embedding(x_cat[:, i]) for i, embedding in enumerate(self.embeddings)]
        return self.mlp(torch.cat(embedded + [x_num], dim=1)).squeeze(1)


embedding_dims = [2, 2, 2, 4]  # regla práctica: dimensiones pequeñas, crecientes con la cardinalidad
for model in (OneHotLogistic(cardinalities, 5), EmbeddingMLP(cardinalities, 5, embedding_dims)):
    print(type(model).__name__, '| parámetros:', sum(p.numel() for p in model.parameters()))

**Resultado e interpretación.** La logística tiene un parámetro por columna; la red tiene más parámetros, pero la mayoría están en las capas densas. Más parámetros no garantizan mejor generalización: lo decide la validación.

### Entrenar con mini-batches y *early stopping*
El ciclo `forward`, pérdida, `zero_grad`, `backward`, `step` queda visible. Tras cada época medimos la pérdida de validación y guardamos el mejor estado; si no mejora durante `patience` épocas, detenemos el entrenamiento.

In [ ]:
def train_model(build_model, epochs=60, lr=0.005, weight_decay=1e-4, batch_size=256, patience=8):
    torch.manual_seed(SEED)  # misma inicialización y orden de batches para cada modelo
    model = build_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loader = DataLoader(TensorDataset(*splits['train']), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    x_cat_val, x_num_val, y_val = splits['val']
    history = {'train_loss': [], 'val_loss': []}
    best_loss, best_state, best_epoch, wait = np.inf, None, 0, 0
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for x_cat, x_num, target in loader:
            loss = F.binary_cross_entropy_with_logits(model(x_cat, x_num), target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(target)
        model.eval()
        with torch.no_grad():
            val_loss = F.binary_cross_entropy_with_logits(model(x_cat_val, x_num_val), y_val).item()
        history['train_loss'].append(total / len(train_idx))
        history['val_loss'].append(val_loss)
        if val_loss < best_loss - 1e-4:
            best_loss, best_state, best_epoch, wait = val_loss, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, history, best_epoch


def predict(model, split):
    x_cat, x_num, _ = splits[split]
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(x_cat, x_num)).cpu().numpy()


trained, histories = {}, {}
for name, build_model in [('logistica_onehot', lambda: OneHotLogistic(cardinalities, 5)),
                          ('mlp_embeddings', lambda: EmbeddingMLP(cardinalities, 5, embedding_dims))]:
    start = time.perf_counter()
    trained[name], histories[name], best_epoch = train_model(build_model)
    print(f'{name}: mejor época {best_epoch} | {time.perf_counter() - start:.1f} s')

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for axis, (name, history) in zip(axes, histories.items()):
    axis.plot(history['train_loss'], label='train')
    axis.plot(history['val_loss'], label='validación')
    axis.set_title(name)
    axis.set_xlabel('época')
axes[0].set_ylabel('entropía cruzada')
axes[0].legend()
plt.tight_layout()
plt.show()

**Resultado e interpretación.** Observa la distancia entre las curvas de train y validación y la época elegida. Si la pérdida de validación sube mientras la de train baja, el modelo empieza a memorizar y el *early stopping* recupera el mejor estado.

### Comparar en validación y evaluar una sola vez en test
Elegimos el modelo con mayor AUC de validación. Solo el modelo elegido se evalúa en test, y el gráfico de ganancia acumulada traduce el resultado a la campaña: qué fracción de los clientes que se van se captura al contactar a la fracción de clientes de mayor riesgo.

In [ ]:
validation = {name: report(churn[val_idx], predict(model, 'val')) for name, model in trained.items()}
for name, metrics in validation.items():
    print(f'{name:18s} validación: {metrics}')

chosen = max(validation, key=lambda name: validation[name]['auc'])
test_metrics = report(churn[test_idx], predict(trained[chosen], 'test'))
print(f'\nModelo elegido por validación: {chosen}')
print('test:', test_metrics, '| lift en el 10 %:', round(test_metrics['precision_top10'] / churn[test_idx].mean(), 2))

def cumulative_gains(y_true, scores):
    captured = np.cumsum(y_true[np.argsort(-scores)]) / y_true.sum()
    return np.arange(1, len(y_true) + 1) / len(y_true), captured

for name, model in trained.items():
    contacted, captured = cumulative_gains(churn[test_idx], predict(model, 'test'))
    plt.plot(contacted, captured, label=name)
plt.plot([0, 1], [0, 1], '--', color='gray', label='aleatorio')
plt.axvline(0.10, color='black', linewidth=0.8)
plt.xlabel('fracción de clientes contactados')
plt.ylabel('fracción de fugas capturadas')
plt.title('Ganancia acumulada en test')
plt.legend()
plt.show()

**Resultado e interpretación.** La línea vertical marca la capacidad de la campaña. La distancia entre cada curva y la diagonal es el valor de priorizar con el modelo. Mostramos ambas curvas en test con fines didácticos, pero la decisión ya se tomó con validación; si la diferencia entre modelos es pequeña, considera la logística por su interpretabilidad y menor costo de mantención.

### Interpretar los embeddings de sucursal
El riesgo de cada sucursal era latente: el modelo solo vio su identificador. Si el embedding capturó ese efecto, alguna dirección del espacio debería correlacionar con `branch_risk`. Proyectamos los 40 vectores a dos dimensiones con SVD.

In [ ]:
mlp = trained['mlp_embeddings']
branch_vectors = mlp.embeddings[3].weight.detach().cpu().numpy()[1:]  # se excluye la fila 0 (desconocido)
centered = branch_vectors - branch_vectors.mean(axis=0)
_, _, components = np.linalg.svd(centered, full_matrices=False)
projection = centered @ components[:2].T

correlations = [abs(np.corrcoef(projection[:, k], branch_risk)[0, 1]) for k in range(2)]
print('|correlación| de cada componente con el riesgo latente:', np.round(correlations, 3))

scatter = plt.scatter(projection[:, 0], projection[:, 1], c=branch_risk, cmap='coolwarm', s=45)
plt.colorbar(scatter, label='riesgo latente de la sucursal')
plt.xlabel('componente 1')
plt.ylabel('componente 2')
plt.title('Embeddings de sucursal proyectados')
plt.show()

**Resultado e interpretación.** Si los colores se ordenan a lo largo de un eje y la correlación es alta, la red recuperó una variable que nadie le entregó explícitamente. Esto es útil (comparte información entre sucursales parecidas) y también un riesgo de gobierno: un embedding puede codificar atributos sensibles indirectos, por lo que debe auditarse como cualquier otra variable.

### ¿Qué ocurre con una sucursal nueva?
En producción aparecerán categorías que no existían en train. Reemplazamos la sucursal de los clientes de test por el índice 0 y comparamos sus predicciones.

In [ ]:
x_cat_test, x_num_test, _ = splits['test']
x_cat_unknown = x_cat_test.clone()
x_cat_unknown[:, 3] = 0
mlp.eval()
with torch.no_grad():
    p_known = torch.sigmoid(mlp(x_cat_test, x_num_test)).cpu().numpy()
    p_unknown = torch.sigmoid(mlp(x_cat_unknown, x_num_test)).cpu().numpy()
print('probabilidad media con sucursal conocida:', round(float(p_known.mean()), 4))
print('probabilidad media con sucursal desconocida:', round(float(p_unknown.mean()), 4))
print('AUC con sucursal desconocida:', round(float(roc_auc(churn[test_idx], p_unknown)), 4))
print('norma del vector "desconocido" frente a la media de sucursales:',
      round(float(np.linalg.norm(mlp.embeddings[3].weight[0].detach().cpu().numpy())), 3), 'vs',
      round(float(np.linalg.norm(branch_vectors, axis=1).mean()), 3))

**Resultado e interpretación.** Ningún ejemplo de train usó la fila 0, así que la pérdida nunca le dio información: solo la movió el *weight decay*, que la empuja desde su inicialización aleatoria hacia el origen. Un vector cercano a cero no equivale a "una sucursal típica" (el promedio de las sucursales no tiene por qué estar en el origen), por lo que las predicciones se desplazan sin justificación. Reservar el índice no basta: hay que **entrenarlo**, por ejemplo reemplazando al azar una fracción de categorías por 0 durante el entrenamiento (desafío de este laboratorio), o asignarle explícitamente el embedding promedio.

## Limitaciones y conclusiones clave
Los resultados dependen de esta semilla, del proceso generador y del protocolo. En datos tabulares reales, los modelos de árboles con *gradient boosting* suelen ser una referencia muy fuerte; una red con embeddings se justifica cuando hay categorías de alta cardinalidad, se quieren reutilizar los embeddings en otros modelos o se combinan datos tabulares con texto o imágenes. No extrapoles métricas sintéticas a datos reales.

- Un embedding es una tabla aprendible que reemplaza al *one-hot* y puede capturar similitud entre categorías.
- Las categorías desconocidas requieren una política explícita y entrenada.
- La métrica debe reflejar la decisión de negocio (capacidad de contacto), no solo la exactitud.

## Ejercicios progresivos
1. **Guiado:** reduce la dimensión del embedding de sucursal a 1 y luego auméntala a 16; compara AUC de validación y la correlación con el riesgo latente.
2. **Independiente:** agrega `pos_weight` a `binary_cross_entropy_with_logits` para compensar el desbalance y compara *log loss* y precisión en el 10 % superior. Explica por qué una métrica empeora y la otra no.
3. **Desafío:** implementa *category dropout*: durante el entrenamiento reemplaza por 0 el 5 % de las sucursales de cada batch. Repite la prueba de sucursal desconocida y compara la caída de AUC.

## Conexión con el siguiente laboratorio
Los embeddings también pueden representar clientes y productos a la vez. En el laboratorio 21 los usamos para construir un sistema de recomendación.